# 12.3 - The AI Gateway Pattern

**Module 12 - Production Deployment** | Netsetos GenAI Engineering

This notebook builds an AI gateway from the inside out - a single OpenAI-compatible front door that owns every LLM call your services make. Across seven keyless, deterministic blocks you assemble the layers a real gateway centralizes: a unified provider facade, a load-balancing router, virtual-key auth, a token-aware rate limiter, config-driven fallbacks, and a per-team cost meter - then map it to the products (LiteLLM, Portkey, Cloudflare, Kong, OpenRouter) that sell it.

Read top to bottom: each code cell has a short **intro above** it and a **step-by-step walkthrough below** it. Run the cell, then check its output against the walkthrough.

## Setup - install dependencies

This lesson is deliberately keyless: every demo uses scripted adapters instead of real SDK calls, so nothing here needs to be installed to follow along. The one optional dependency is `python-dotenv`, handy only if you want to load real keys from a `.env` file when you later point this at a live provider.

In [ ]:
# Install Python dependencies used by this lesson (uncomment on Colab / fresh env).
# !pip install python-dotenv -q  # noqa


**Explanation**

A one-line environment-prep cell. The install is commented out because none of the runnable blocks in this notebook make a network call - the pip line is there only for a fresh Colab session that wants to load keys from a file.

**How the code works - step by step**
- **`!pip install python-dotenv -q`** - commented out; uncomment on Colab or a fresh env only if you plan to load real provider keys from a `.env` file.

**In one line:** nothing to install to run this notebook - it is fully scripted.

**What you'll see:** (no output - environment prep)

## Setup - declare the provider key slots

A gateway holds one key per provider; the app never touches them. This cell just declares the three slots (OpenAI, Anthropic, Google) as empty environment variables so the rest of the notebook can reference them without crashing. You do not need to fill any in - every demo below is scripted.

In [ ]:
import os
# Load API keys from the environment (or a .env file via python-dotenv).
# Never hardcode keys. Any ONE provider key is enough to start;
# the multi-provider demos are best with two or more.
os.environ.setdefault("OPENAI_API_KEY", "")     # platform.openai.com
os.environ.setdefault("ANTHROPIC_API_KEY", "")    # console.anthropic.com
os.environ.setdefault("GOOGLE_API_KEY", "")     # aistudio.google.com


**Explanation**

Environment prep, not a model call. It uses `setdefault` so it only sets a blank value when the variable is not already present, leaving any real key you exported untouched. It also names, in comments, where each real key comes from.

**How the code works - step by step**
- **`import os`** - gives access to the process environment.
- **`os.environ.setdefault("OPENAI_API_KEY", "")`** and the two siblings - register empty slots for OpenAI, Anthropic, and Google without overwriting anything you already set.
- The trailing comments point at each provider's console (platform.openai.com, console.anthropic.com, aistudio.google.com).

**In one line:** declare the provider key slots the gateway would own - blank here, because everything runs scripted.

**What you'll see:** (no output - environment prep)

## 1 - One front door for every provider (the unified facade)

The move that makes everything else possible: wrap OpenAI, Anthropic, and Google behind a **single OpenAI-compatible call** so your app never imports a provider SDK. Once every call flows through one function, the gateway can own auth, routing, limits, cost, and logging in one place - and a model swap becomes a config change, not a deploy across every service.

In [ ]:
# ONE FRONT DOOR - a gateway wraps every provider behind a SINGLE OpenAI-compatible interface,
# so your app calls one function and a model swap becomes a config change, not a code change.
# Scripted adapters stand in for the real SDKs, so this runs with no key.

# Each provider adapter normalizes its own API into the SAME response shape.
def openai_adapter(model, prompt):    return {"text": f"[openai:{model}] {prompt[:18]}...", "provider": "openai"}
def anthropic_adapter(model, prompt): return {"text": f"[anthropic:{model}] {prompt[:18]}...", "provider": "anthropic"}
def gemini_adapter(model, prompt):    return {"text": f"[gemini:{model}] {prompt[:18]}...", "provider": "gemini"}

# The gateway's model -> provider map is CONFIG (not code).
MODEL_ROUTES = {
    "gpt-5":             openai_adapter,
    "claude-sonnet-4-6": anthropic_adapter,
    "gemini-2.5-flash":  gemini_adapter,
}

def gateway_chat(model, prompt):          # the ONE call every app makes
    adapter = MODEL_ROUTES[model]
    return adapter(model, prompt)

print("Three providers, one OpenAI-compatible call:")
for model in ["gpt-5", "claude-sonnet-4-6", "gemini-2.5-flash"]:
    r = gateway_chat(model, "Summarize the refund policy")
    print(f"  gateway_chat({model:18}) -> provider={r['provider']:9} {r['text']}")

# A model swap is a one-line config change - the app code never changes.
DEFAULT_MODEL = "gpt-5"
print(f"\nApp calls gateway_chat(DEFAULT_MODEL): default is {DEFAULT_MODEL}")
DEFAULT_MODEL = "gemini-2.5-flash"        # swap to a cheaper model in CONFIG
print(f"Swap DEFAULT_MODEL -> {DEFAULT_MODEL}: same app code, now routed to {gateway_chat(DEFAULT_MODEL, 'hi')['provider']}")
# Output:
# Three providers, one OpenAI-compatible call:
#   gateway_chat(gpt-5             ) -> provider=openai    [openai:gpt-5] Summarize the refu...
#   gateway_chat(claude-sonnet-4-6 ) -> provider=anthropic [anthropic:claude-sonnet-4-6] Summarize the refu...
#   gateway_chat(gemini-2.5-flash  ) -> provider=gemini    [gemini:gemini-2.5-flash] Summarize the refu...
#
# App calls gateway_chat(DEFAULT_MODEL): default is gpt-5
# Swap DEFAULT_MODEL -> gemini-2.5-flash: same app code, now routed to gemini

**Explanation**

This block builds a facade over three providers using scripted adapters that each return the same response shape, so it runs with no key. The core idea: a single routing table plus a single `gateway_chat` entry point turns an N-apps-by-M-providers tangle into N-apps-to-one-door. Note it normalizes only the response shape to stay keyless - a real facade must also translate the request across providers.

**How the code works - step by step**
- **`openai_adapter` / `anthropic_adapter` / `gemini_adapter`** - three stand-in adapters, each normalizing its provider into the *same* `{text, provider}` dict.
- **`MODEL_ROUTES`** - maps a model name to its adapter; this routing table is **config**, not code.
- **`gateway_chat(model, prompt)`** - the one call every app makes; it looks up the adapter and delegates.
- **The swap demo** - reassigning `DEFAULT_MODEL` from `gpt-5` to `gemini-2.5-flash` re-routes to a different provider with zero app-code change.

**In one line:** one table + one call = every provider behind a single door, model swap by config.

**What you'll see:** Prints three provider calls all made through the same `gateway_chat`, then shows `DEFAULT_MODEL` flipping from gpt-5 (openai) to gemini-2.5-flash (gemini) with the app code untouched.

## 2 - Routing and load balancing

With every call flowing through the front door, the gateway can spread it across a **model group** - several deployments of the same logical model - by a strategy: priority order, weighted share, or lowest latency. And when a deployment returns a 429, it goes on **cooldown** so traffic shifts off it until it recovers.

In [ ]:
# ROUTING & LOAD BALANCING - a gateway routes a request across a MODEL GROUP (several deployments
# of the same logical model) by a strategy: priority order, weighted, or least-latency. A 429 puts a
# deployment on cooldown so traffic shifts off it. Deterministic, no key.

# A model group: deployments with an order (lower = higher priority), a weight, and a latency (ms).
deployments = [
    {"id": "openai-key-1",  "order": 1, "weight": 1, "latency": 800, "cooling": False},
    {"id": "azure-key-2",   "order": 2, "weight": 5, "latency": 500, "cooling": False},
    {"id": "openai-key-3",  "order": 3, "weight": 1, "latency": 200, "cooling": False},
]
def available(deps): return [d for d in deps if not d["cooling"]]

def route(deps, strategy):
    live = available(deps)
    if strategy == "priority":   return min(live, key=lambda d: d["order"])
    if strategy == "weighted":   return max(live, key=lambda d: d["weight"])   # highest weight = biggest share
    if strategy == "latency":    return min(live, key=lambda d: d["latency"])

print("Same request, three routing strategies pick three different deployments:")
for s in ["priority", "weighted", "latency"]:
    print(f"  {s:9} -> {route(deployments, s)['id']}")

# A 429 on the top-priority deployment puts it on COOLDOWN; priority then shifts to the next.
print("\nopenai-key-1 returns 429 -> placed on cooldown:")
deployments[0]["cooling"] = True
print(f"  priority now -> {route(deployments, 'priority')['id']}  (traffic shifted off the cooling deployment)")
# Output:
# Same request, three routing strategies pick three different deployments:
#   priority  -> openai-key-1
#   weighted  -> azure-key-2
#   latency   -> openai-key-3
#
# openai-key-1 returns 429 -> placed on cooldown:
#   priority now -> azure-key-2  (traffic shifted off the cooling deployment)

**Explanation**

A deterministic routing demo, no key. Three deployments differ in order, weight, and latency; a single `route` function picks one according to whichever strategy you name, and a `cooling` flag pulls a throttled deployment out of the live set. Read it as: the strategy is a config choice that changes how traffic spreads without touching any service.

**How the code works - step by step**
- **`deployments`** - a model group of three, each with `order` (lower = higher priority), `weight`, `latency`, and a `cooling` flag.
- **`available`** - filters out any deployment currently on cooldown.
- **`route(deps, strategy)`** - `priority` picks the lowest `order`, `weighted` picks the highest `weight`, `latency` picks the lowest `latency`.
- **The 429 case** - setting `openai-key-1["cooling"] = True` drops it from the live set, so priority routing shifts to the next available deployment.

**In one line:** route a model group by strategy; a 429 cools a backend and traffic moves on.

**What you'll see:** Shows the same request landing on three different deployments (priority->openai-key-1, weighted->azure-key-2, latency->openai-key-3), then priority shifting to azure-key-2 after openai-key-1 is put on cooldown.

## 3 - Virtual keys and auth

The security move. Instead of copying the real provider key into every service, the gateway holds the real keys in **one** place and issues each app a **scoped virtual key** with its own model allow-list and budget. A request is authorized against the virtual key - wrong model, over budget, or unknown key is rejected before any real call - and you can revoke one app without touching the others.

In [ ]:
# VIRTUAL KEYS - the gateway holds the REAL provider keys and hands each app a scoped VIRTUAL key
# with its own model allow-list and budget. A request is authorized against the virtual key; the real
# keys never leave the gateway. Scripted, deterministic, no key.

# The gateway's secret store (real keys) - NEVER returned to any app.
_REAL_KEYS = {"openai": "sk-real-openai-...", "anthropic": "sk-real-anthropic-..."}

# Virtual keys issued to teams: what each is allowed to do.
virtual_keys = {
    "vk-team-support": {"allowed": {"gpt-5", "gemini-2.5-flash"}, "budget_left": 5.00},
    "vk-team-intern":  {"allowed": {"gemini-2.5-flash"},          "budget_left": 0.00},
}

def authorize(vkey, model, est_cost):
    k = virtual_keys.get(vkey)
    if k is None:                    return False, "unknown virtual key"
    if model not in k["allowed"]:    return False, f"model {model} not in this key's allow-list"
    if est_cost > k["budget_left"]:  return False, f"over budget (${k['budget_left']:.2f} left)"
    return True, "authorized"

print("Authorize requests against virtual keys (real provider keys stay in the gateway):")
tests = [("vk-team-support", "gpt-5", 0.20),          # allowed + in budget
         ("vk-team-support", "claude-sonnet-4-6", 0.20),  # model not allowed
         ("vk-team-intern",  "gemini-2.5-flash", 0.20)]   # over budget ($0 left)
for vkey, model, cost in tests:
    ok, why = authorize(vkey, model, cost)
    print(f"  {vkey:16} {model:18} -> {'PASS' if ok else 'REJECT'} ({why})")
print(f"\nThe real keys ({list(_REAL_KEYS)}) were never exposed to any app - only virtual keys are.")
# Output:
# Authorize requests against virtual keys (real provider keys stay in the gateway):
#   vk-team-support  gpt-5              -> PASS (authorized)
#   vk-team-support  claude-sonnet-4-6  -> REJECT (model claude-sonnet-4-6 not in this key's allow-list)
#   vk-team-intern   gemini-2.5-flash   -> REJECT (over budget ($0.00 left))
#
# The real keys (['openai', 'anthropic']) were never exposed to any app - only virtual keys are.

**Explanation**

A scripted authorization layer, no key. A private `_REAL_KEYS` store never leaves the gateway; a `virtual_keys` table records what each app may do; and `authorize` checks the key, the model allow-list, and the budget in order. This is the credential-proxy idea from 11.11, productized: the real keys stay behind the desk and apps only ever hold badges.

**How the code works - step by step**
- **`_REAL_KEYS`** - the gateway's secret store of real provider keys, never returned to any app.
- **`virtual_keys`** - two issued keys, each with an `allowed` model set and a `budget_left`.
- **`authorize(vkey, model, est_cost)`** - rejects an unknown key, then a model not in the allow-list, then a call over the remaining budget; otherwise authorizes.
- **The three tests** - an allowed+in-budget call passes; a non-allow-listed model is rejected; the intern key ($0 left) is rejected on budget.

**In one line:** authorize against a scoped virtual key so the real keys never leave the gateway.

**What you'll see:** Prints PASS for vk-team-support on gpt-5, REJECT for the same key on a non-allow-listed model, and REJECT for the over-budget intern key - then confirms the real keys were never exposed.

## 4 - Rate limiting: requests AND tokens

The trap teams hit coming from ordinary web APIs. A normal API limits requests per minute (RPM), but LLM providers charge by the token and limit tokens per minute (TPM) - and TPM is the load-bearing one. A single 200,000-token call costs about as much as fifty ordinary calls, and a request-only limiter waves that whale straight through. So a gateway must enforce **both** axes.

In [ ]:
# RATE LIMITING - LLM limits are on TWO axes: requests-per-minute (RPM) AND tokens-per-minute (TPM).
# Token limits are the load-bearing ones: one long-context call can blow the TPM budget while sailing
# past a request-only limiter. A gateway must enforce BOTH. Deterministic, no key.

RPM_LIMIT, TPM_LIMIT = 4, 100_000
reqs, toks = 0, 0

def allow(n_tokens, token_aware=True):
    global reqs, toks
    reqs += 1; toks += n_tokens
    if reqs > RPM_LIMIT:                        return False, f"RPM exceeded ({reqs} > {RPM_LIMIT})"
    if token_aware and toks > TPM_LIMIT:        return False, f"TPM exceeded ({toks:,} > {TPM_LIMIT:,})"
    return True, "allowed"

print("A burst of 5 small calls (500 tokens each) trips the RPM limit:")
for i in range(5):
    ok, why = allow(500)
    print(f"  call {i+1}: 500 tokens -> {'allowed' if ok else 'BLOCKED: ' + why}")

# Reset, then send ONE 200,000-token 'whale' call.
reqs, toks = 0, 0
print("\nOne 200,000-token call - the request-only limiter MISSES it, the token-aware one catches it:")
ok_req, _ = allow(200_000, token_aware=False)      # RPM-only: 1 request, looks fine
reqs, toks = 0, 0
ok_tok, why = allow(200_000, token_aware=True)      # RPM + TPM: the tokens blow the budget
print(f"  request-only limiter: {'allowed' if ok_req else 'BLOCKED'}  <- BUG: 1 call = 1 request, so it passes")
print(f"  token-aware limiter:  {'allowed' if ok_tok else 'BLOCKED'}  <- {why} (one whale = 2x the entire TPM budget)")
# Output:
# A burst of 5 small calls (500 tokens each) trips the RPM limit:
#   call 1: 500 tokens -> allowed
#   call 2: 500 tokens -> allowed
#   call 3: 500 tokens -> allowed
#   call 4: 500 tokens -> allowed
#   call 5: 500 tokens -> BLOCKED: RPM exceeded (5 > 4)
#
# One 200,000-token call - the request-only limiter MISSES it, the token-aware one catches it:
#   request-only limiter: allowed  <- BUG: 1 call = 1 request, so it passes
#   token-aware limiter:  BLOCKED  <- TPM exceeded (200,000 > 100,000) (one whale = 2x the entire TPM budget)

**Explanation**

A deterministic two-axis limiter, no key. Running counters for requests and tokens are checked against `RPM_LIMIT` and `TPM_LIMIT`; a `token_aware` flag lets the demo show the request-only limiter failing where the token-aware one succeeds. The point: the two axes catch different abuses - a burst of tiny calls, and one enormous one.

**How the code works - step by step**
- **`RPM_LIMIT, TPM_LIMIT` + `reqs, toks`** - the two ceilings and their running meters.
- **`allow(n_tokens, token_aware=True)`** - increments both meters; blocks if `reqs` exceeds RPM, and (when token-aware) if `toks` exceeds TPM.
- **The burst** - five 500-token calls trip the RPM limit on call five.
- **The whale** - one 200,000-token call passes the `token_aware=False` limiter (the bug: it is one request) but is blocked by the token-aware limiter because the tokens exceed TPM.

**In one line:** enforce RPM and TPM - a burst trips requests, a whale trips tokens.

**What you'll see:** Prints four allowed small calls then a BLOCKED fifth (RPM exceeded 5>4), and shows the 200,000-token call allowed by the request-only limiter but BLOCKED (TPM exceeded 200,000>100,000) by the token-aware one.

## 5 - Fallbacks and retries as config

In 12.2 you hand-coded a resilient client. A gateway lets you write that reliability **once**, as configuration. The failover rules live in a versioned config file: an **order chain** within a model group, then a **cross-model fallback** when the whole group is exhausted - so the reliability spine protects every app behind the gateway with no per-service retry code.

In [ ]:
# FALLBACKS AS CONFIG - 12.2's reliability patterns become DECLARATIVE gateway config: an order chain
# within a model group, plus cross-model fallbacks, in a versioned config file. One config protects
# every app - no per-app retry code. Scripted, deterministic, no key.

# The gateway config (what you version in your repo, not code sprinkled through every service).
CONFIG = {
    "model_groups": {
        "chat": [
            {"deployment": "gpt-5@openai",  "order": 1},
            {"deployment": "gpt-5@azure",   "order": 2},   # same model, second key
        ]
    },
    "fallbacks": {"chat": ["claude-sonnet-4-6"]},           # cross-model fallback when the group is exhausted
}

def call(deployment, down):  return "ok" if deployment not in down else "429"

def route_with_fallback(config, group, down):
    for dep in sorted(config["model_groups"][group], key=lambda d: d["order"]):   # try the order chain
        if call(dep["deployment"], down) == "ok":
            return dep["deployment"], "primary group"
    for fb in config["fallbacks"].get(group, []):                                  # then cross-model fallbacks
        if call(fb, down) == "ok":
            return fb, "cross-model fallback"
    return None, "all failed"

print("Config-driven failover (no per-app code):")
served, via = route_with_fallback(CONFIG, "chat", down=set())
print(f"  everything healthy      -> served by {served} ({via})")
served, via = route_with_fallback(CONFIG, "chat", down={"gpt-5@openai"})
print(f"  gpt-5@openai 429        -> served by {served} ({via})")
served, via = route_with_fallback(CONFIG, "chat", down={"gpt-5@openai", "gpt-5@azure"})
print(f"  whole gpt-5 group down  -> served by {served} ({via})")
print("\nChanging the failover behavior is a CONFIG edit, not a redeploy of every service.")
# Output:
# Config-driven failover (no per-app code):
#   everything healthy      -> served by gpt-5@openai (primary group)
#   gpt-5@openai 429        -> served by gpt-5@azure (primary group)
#   whole gpt-5 group down  -> served by claude-sonnet-4-6 (cross-model fallback)
#
# Changing the failover behavior is a CONFIG edit, not a redeploy of every service.

**Explanation**

A config-driven failover demo, no key. A `CONFIG` object declares the model group's order chain and its cross-model fallback; `route_with_fallback` walks the order chain first, then the fallbacks, given a set of down deployments. Read it as: 12.2's reliability patterns expressed declaratively, edited in one file instead of copy-pasted into five services.

**How the code works - step by step**
- **`CONFIG`** - a `chat` model group with two ordered deployments plus a `fallbacks` entry to `claude-sonnet-4-6`.
- **`call(deployment, down)`** - returns `429` if the deployment is in the `down` set, else `ok`.
- **`route_with_fallback`** - tries the order chain in `order` sequence, then the cross-model fallbacks, returning the first that answers.
- **The three scenarios** - all healthy serves the order-1 deployment; a 429 on the primary shifts to order-2 in the same group; the whole group down falls through to the cross-model fallback.

**In one line:** an order chain plus cross-model fallback drives failover from one versioned config.

**What you'll see:** Prints served-by gpt-5@openai when healthy, gpt-5@azure when the primary 429s, and claude-sonnet-4-6 (cross-model fallback) when the whole gpt-5 group is down - all without app code.

## 6 - Cost, budgets and observability

Because every call flows through the gateway, it is the one place that can meter cost. It computes each call as **tokens times price**, attributes it per team, enforces a **budget** (rejecting a call that would breach the cap before it runs), and emits a structured log line. This is the answer to the cold-open's unanswerable question: who spent what becomes a query, not a forensic reconstruction.

In [ ]:
# COST, BUDGETS & OBSERVABILITY - the gateway meters every call as tokens x price, attributes it per
# team, enforces a budget, and logs a structured line. One place for spend across all providers.
# Prices are illustrative ($/1k tokens). Deterministic, no key.

PRICE = {"gpt-5": 0.005, "claude-sonnet-4-6": 0.003, "gemini-2.5-flash": 0.0004}   # $ per 1k tokens (illustrative)
team_spend = {"support": 0.0}
TEAM_BUDGET = 1.00                                                                  # monthly cap (illustrative)

def meter(team, model, tokens):
    cost = round(tokens / 1000 * PRICE[model], 4)
    if team_spend[team] + cost > TEAM_BUDGET:
        return None, f"BLOCKED: team '{team}' would exceed its ${TEAM_BUDGET:.2f} budget"
    team_spend[team] += cost
    log = {"team": team, "model": model, "tokens": tokens, "cost": cost, "spend_after": round(team_spend[team], 4)}
    return log, "ok"

print("Meter each call, attribute per team, enforce the budget, log a structured line:")
calls = [("support", "gpt-5", 60_000), ("support", "gpt-5", 60_000),
         ("support", "gpt-5", 60_000), ("support", "gpt-5", 60_000)]   # 4 x $0.30 = $1.20 > $1.00
for team, model, tokens in calls:
    log, status = meter(team, model, tokens)
    if log:
        print(f"  {log}")
    else:
        print(f"  {status}  (spend so far ${team_spend[team]:.2f})")
print(f"\nOne dashboard answers 'who spent what': support = ${team_spend['support']:.2f}, and the 4th call was denied.")
# Output:
# Meter each call, attribute per team, enforce the budget, log a structured line:
#   {'team': 'support', 'model': 'gpt-5', 'tokens': 60000, 'cost': 0.3, 'spend_after': 0.3}
#   {'team': 'support', 'model': 'gpt-5', 'tokens': 60000, 'cost': 0.3, 'spend_after': 0.6}
#   {'team': 'support', 'model': 'gpt-5', 'tokens': 60000, 'cost': 0.3, 'spend_after': 0.9}
#   BLOCKED: team 'support' would exceed its $1.00 budget  (spend so far $0.90)
#
# One dashboard answers 'who spent what': support = $0.90, and the 4th call was denied.

**Explanation**

A scripted cost meter, no key. A price table and a per-team spend store back a single `meter` function that prices a call, checks it against the team budget, and either records+logs it or blocks it. The idea: real-time per-team attribution and a hard budget cap, in one place, across all providers.

**How the code works - step by step**
- **`PRICE` / `team_spend` / `TEAM_BUDGET`** - illustrative $/1k-token prices, the running per-team spend, and the monthly cap.
- **`meter(team, model, tokens)`** - computes `tokens/1000 * price`; if the new total would exceed the budget it blocks, otherwise it adds to spend and returns a structured log dict.
- **The four calls** - four $0.30 gpt-5 calls (totaling $1.20 against a $1.00 cap): the first three log and accumulate, the fourth is blocked before running.

**In one line:** meter tokens x price per team, enforce the budget, log every call.

**What you'll see:** Prints three structured log lines with spend climbing 0.3 -> 0.6 -> 0.9, then a BLOCKED message for the fourth call, and a summary that support spent $0.90 with the 4th call denied.

## 7 - The landscape: build vs buy

Having built a gateway from the inside, you can now read the 2026 product landscape, which splits on **self-hosted vs managed**. This block routes a team's real needs - existing Kong mesh, guardrails/semantic cache, Cloudflare ecosystem, full control, or one-key simplicity - to the tool that fits: LiteLLM, Portkey, Cloudflare AI Gateway, Kong, or OpenRouter.

In [ ]:
# THE LANDSCAPE - build vs buy. A gateway is either self-hosted (you own it) or managed (zero ops).
# Route a team's needs to the right tool. Deterministic, no key.

def choose(self_host, needs_guardrails_or_semantic_cache, has_kong_mesh, in_cloudflare, wants_one_key):
    if has_kong_mesh:                         return "Kong AI Gateway (LLM routing inside your existing Kong mesh)"
    if needs_guardrails_or_semantic_cache:    return "Portkey (guardrails + semantic caching + observability, mostly SaaS)"
    if in_cloudflare:                         return "Cloudflare AI Gateway (fully managed, edge caching, zero ops)"
    if self_host:                             return "LiteLLM (self-hosted OpenAI-compatible proxy, budgets per key)"
    if wants_one_key:                         return "OpenRouter (managed unified API, one key over 300+ models)"
    return "LiteLLM (sensible open-source default)"

cases = [
    ("full control, self-host everything",        dict(self_host=True,  needs_guardrails_or_semantic_cache=False, has_kong_mesh=False, in_cloudflare=False, wants_one_key=False)),
    ("need guardrails + semantic caching",        dict(self_host=False, needs_guardrails_or_semantic_cache=True,  has_kong_mesh=False, in_cloudflare=False, wants_one_key=False)),
    ("already run a Kong API mesh",               dict(self_host=False, needs_guardrails_or_semantic_cache=False, has_kong_mesh=True,  in_cloudflare=False, wants_one_key=False)),
    ("live in Cloudflare, want zero ops",         dict(self_host=False, needs_guardrails_or_semantic_cache=False, has_kong_mesh=False, in_cloudflare=True,  wants_one_key=False)),
    ("just want one key over every model",        dict(self_host=False, needs_guardrails_or_semantic_cache=False, has_kong_mesh=False, in_cloudflare=False, wants_one_key=True)),
]
print("Route each team's needs to a gateway:")
for name, props in cases:
    print(f"  {name:42} -> {choose(**props)}")
print("\nBuild vs buy: self-host for control (LiteLLM); buy for guardrails/caching/zero-ops (Portkey/Cloudflare/OpenRouter).")
# Output:
# Route each team's needs to a gateway:
#   full control, self-host everything         -> LiteLLM (self-hosted OpenAI-compatible proxy, budgets per key)
#   need guardrails + semantic caching         -> Portkey (guardrails + semantic caching + observability, mostly SaaS)
#   already run a Kong API mesh                -> Kong AI Gateway (LLM routing inside your existing Kong mesh)
#   live in Cloudflare, want zero ops          -> Cloudflare AI Gateway (fully managed, edge caching, zero ops)
#   just want one key over every model         -> OpenRouter (managed unified API, one key over 300+ models)
#
# Build vs buy: self-host for control (LiteLLM); buy for guardrails/caching/zero-ops (Portkey/Cloudflare/OpenRouter).

**Explanation**

A deterministic decision function, no key. `choose` encodes a priority-ordered set of if-checks that map a team's constraints to a recommended gateway, and a table of five cases exercises each branch. Read it as a decision aid, not a model call: self-host for control, buy for guardrails, caching, or zero ops.

**How the code works - step by step**
- **`choose(...)`** - checks constraints in priority order: Kong mesh first, then guardrails/semantic-cache (Portkey), then Cloudflare, then self-host (LiteLLM), then one-key (OpenRouter), with a LiteLLM default.
- **`cases`** - five teams, each a dict of the five boolean needs.
- **The loop** - runs `choose(**props)` for each and prints the matched tool.

**In one line:** map a team's needs to a gateway - build (LiteLLM) for control, buy for guardrails/caching/zero-ops.

**What you'll see:** Prints each of the five teams routed to its tool - LiteLLM, Portkey, Kong, Cloudflare, OpenRouter respectively - then the build-vs-buy summary line.

## 7b - The same thing as a real LiteLLM config

You do not hand-roll the gateway in production; you run LiteLLM (or Portkey / Cloudflare) and put routing, keys, limits, and fallbacks in **one versioned config file**. This illustrative block shows a minimal `config.yaml` that centralizes everything you built by hand in Steps 2, 3, and 5.

In [ ]:
# THE PRODUCTION GATEWAY - a LiteLLM config.yaml (illustrative minimal example).
# You do not hand-roll the gateway in production; you run LiteLLM (or Portkey / Cloudflare) and put
# routing, keys, limits, and fallbacks in ONE versioned config file. App code just calls the proxy.

# config.yaml
CONFIG = """
model_list:                                    # a model GROUP: two deployments of one logical model
  - model_name: chat
    litellm_params: {model: openai/gpt-5, api_key: os.environ/OPENAI_KEY, rpm: 500}
  - model_name: chat
    litellm_params: {model: azure/gpt-5, api_key: os.environ/AZURE_KEY, rpm: 500}

router_settings:
  routing_strategy: latency-based-routing       # or simple-shuffle / least-busy / usage-based-routing
  num_retries: 3
  fallbacks: [{chat: [claude-sonnet-4-6]}]       # cross-model fallback when the group is exhausted

general_settings:
  master_key: os.environ/LITELLM_MASTER_KEY      # mint scoped virtual keys via POST /key/generate
"""
# Start the proxy:   litellm --config config.yaml         (a Docker container + a small Postgres)
# Apps call the OpenAI-compatible endpoint with a VIRTUAL key, never a real provider key:
#   from openai import OpenAI
#   client = OpenAI(base_url="http://gateway:4000", api_key="sk-virtual-team-support")
#   client.chat.completions.create(model="chat", messages=[...])
# Output: (illustrative minimal example - needs `pip install litellm[proxy]` + provider keys in env.)
# One config.yaml centralizes routing, retries, fallbacks, virtual keys, and budgets for every app.

**Explanation**

A reference cell, not a runnable demo - it holds a `config.yaml` as a string and shows the client code in comments. Nothing executes a call; it maps each hand-built layer to its one-line config equivalent so you recognize the real product when you meet it.

**How the code works - step by step**
- **`model_list`** - two deployments of the logical model `chat` (openai + azure) - Step 2's model group, as config.
- **`router_settings`** - `routing_strategy`, `num_retries`, and `fallbacks` - Steps 2 and 5 in three lines.
- **`general_settings.master_key`** - mints scoped virtual keys via `POST /key/generate` - Step 3.
- **The commented client code** - apps call the OpenAI-compatible endpoint with a *virtual* key, never a real provider key.

**In one line:** one config.yaml centralizes routing, retries, fallbacks, and virtual keys for every app.

**What you'll see:** No live output - it is an illustrative config string (running it needs `pip install litellm[proxy]` plus real provider keys). The takeaway comment: one config.yaml centralizes everything the earlier cells built by hand.

You have now built every layer a gateway centralizes - facade, router, virtual keys, RPM+TPM limits, config fallbacks, and a cost meter - and seen the same ideas expressed as a real LiteLLM `config.yaml`. The through-line: because every call flows through one front door, cross-cutting concerns (auth, routing, limits, reliability, cost, logging) live in one place as configuration instead of being re-implemented in every service. The next layer a gateway adds is caching (prompt, semantic, edge) in Lesson 12.4; monitoring its structured logs and per-team cost is Lesson 12.8; the deep eval/observability stack it feeds is Module 14.